In [ ]:
# Analyze top N dependents of pymatgen (sorted by GitHub repo stars)

In [ ]:
from pathlib import Path

CSV_FILE = "pymatgen_dependents_with_stars.csv"
DEST_DIR = Path("dependent-repos")
TOP_X: int = 10
MIN_USAGE: int = 10  # filter by minimum API usage by dependent

SKIP_REPOS: set[str] = {
    "dscribe",  # only uses pmg in tests
    "Uni-Mol",  # no pmg usage
    "matgenb",  # docs of pmg
    "api",  # somewhat internal to MP
    "MaterialsInformatics",  # course materials
    "pyxem",  # no pmg usage
    "streamlit-file-browser",  # a browser used pmg once to visualize Structure
    "psiflow",  # no pmg usage
    "PiNN",  # no pmg usage
    "matsciml",  # repo removed
    # The following is filtered by minimum API usage
    "mat2vec",  # 3 usage
    "mattersim",  # 8 usage
    "DeepH-pack",  # 8 usage
    "materials",  # 1 usage
    "gptchem",  # 9 usage
    "automatminer",  # 4 usage
    "XenonPy",  # 6 usage
    "ORGANIC",  # 2 usage
    "MAST-ML",  # 5 usage
}

In [ ]:
# Clone the top X repos

import csv
import subprocess

DEST_DIR.mkdir(parents=True, exist_ok=True)

with open(CSV_FILE, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows = list(reader)[: TOP_X + len(SKIP_REPOS)]

repos: list[str] = []  # used to preserve repo order (by stars)

for row in rows:
    repo_name = row["owner_repo"].split("/")[-1]
    repo_url = row["repo_url"]

    if repo_name in SKIP_REPOS:
        continue
    else:
        print(repo_name)

    repos.append(repo_name)

    dest_path = DEST_DIR / repo_name
    if dest_path.is_dir():
        print(f"✅ Skipping {repo_name} (already cloned)")
        continue

    print(f"⬇️  Cloning {repo_name} ...")

    subprocess.run(
        ["git", "clone", "--depth", "1", repo_url, str(dest_path)],
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    print(f"✅ Done: {repo_name}")

materials_discovery
✅ Skipping materials_discovery (already cloned)
AIRS
✅ Skipping AIRS (already cloned)
megnet
✅ Skipping megnet (already cloned)
matgl
✅ Skipping matgl (already cloned)
PaddleScience
✅ Skipping PaddleScience (already cloned)
PyXtal
✅ Skipping PyXtal (already cloned)
pymatviz
✅ Skipping pymatviz (already cloned)
all-atom-diffusion-transformer
✅ Skipping all-atom-diffusion-transformer (already cloned)
quacc
✅ Skipping quacc (already cloned)
doped
✅ Skipping doped (already cloned)
matbench-discovery
✅ Skipping matbench-discovery (already cloned)
CrystaLLM
✅ Skipping CrystaLLM (already cloned)


In [ ]:
# Build an exclusion directory map for each repo

from dataclasses import dataclass


@dataclass
class RepoConfig:
    exclude: list[str] | None = None


# Default exclude list for all repos
DEFAULT_EXCLUDE = ["tests", "docs", ".github"]

REPO_PATH_MAP: dict[str, RepoConfig] = {
    name: RepoConfig(exclude=DEFAULT_EXCLUDE) for name in repos
}

In [ ]:
# Install api-analyzer
api_analyzer_path = Path().resolve() / "api-analyzer"
!uv pip install "{api_analyzer_path}"

Using Python 3.14.3 environment at: /Users/yang/developer/pymatgen-2-paper/.venv
Resolved 1 package in 15ms                                           
   Building api-analyzer @ file:///Users/yang/developer/pymatgen-2-paper/fig_scr
   Building api-analyzer @ file:///Users/yang/developer/pymatgen-2-paper/fig_scr
   Building api-analyzer @ file:///Users/yang/developer/pymatgen-2-paper/fig_scr
   Building api-analyzer @ file:///Users/yang/developer/pymatgen-2-paper/fig_scr
   Building api-analyzer @ file:///Users/yang/developer/pymatgen-2-paper/fig_scr
      Built api-analyzer @ file:///Users/yang/developer/pymatgen-2-paper/fig_scr
Prepared 1 package in 916ms                                              
Uninstalled 1 package in 1ms
Installed 1 package in 1ms (from file:///Users/yang/develope
 ~ api-analyzer==0.1.0 (from file:///Users/yang/developer/pymatgen-2-paper/fig_scripts/pmg-api-usage-dependent-dependency/api-analyzer)


In [ ]:
from api_analyzer import analyze_paths

results: dict[str, dict[str, dict[str, int]]] = {}

for repo_name, cfg in REPO_PATH_MAP.items():
    repo_path = DEST_DIR / repo_name
    if not repo_path.exists():
        raise NotADirectoryError(f"{repo_name}: path not found")

    print(f"🔍 Analyzing {repo_name} for pymatgen usage...")

    aliases, usage = analyze_paths(
        repo_path,
        package="pymatgen",
        exclude=cfg.exclude or [],
    )

    results[repo_name] = {
        # "aliases": aliases,
        "usage": usage,
    }

    if not usage:
        raise RuntimeError(f"No pymatgen usage found in {repo_name}")

    total_usage = sum(usage.values())
    if total_usage < MIN_USAGE:
        raise RuntimeError(
            f"Not enough pmg usage in {repo_name}, found {total_usage} expect at least {MIN_USAGE}"
        )

print("✅ Analysis finished.")

🔍 Analyzing materials_discovery for pymatgen usage...
🔍 Analyzing AIRS for pymatgen usage...


<unknown>:77: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
<unknown>:123: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
<unknown>:287: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:221: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<unknown>:80: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<unknown>:23: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<unknown>:26: SyntaxWarning: "\

⚠️  Skipping dependent-repos/AIRS/OpenMI/EquiGX/EquiGX/SCOP.ipynb (AST parse error: invalid syntax (<unknown>, line 17))
⚠️  Skipping dependent-repos/AIRS/OpenMI/EquiGX/EquiGX/BioLip.ipynb (AST parse error: invalid decimal literal (<unknown>, line 2446))


<unknown>:318: SyntaxWarning: "\o" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\o"? A raw string is also an option.
<unknown>:618: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:980: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:1370: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:1681: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:176: SyntaxWarning: "\#" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\#"? A raw string is also an option.
<unknown>:154: SyntaxWarni

⚠️  Skipping dependent-repos/AIRS/OpenProt/LatentDiff/LatentDiff/data/gen_data_for_diffusion.ipynb (AST parse error: invalid syntax (<unknown>, line 48))
⚠️  Skipping dependent-repos/AIRS/OpenProt/LatentDiff/LatentDiff/ProteinMPNN/colab_notebooks/quickdemo_wAF2.ipynb (AST parse error: expected an indented block after 'if' statement on line 246 (<unknown>, line 247))
🔍 Analyzing megnet for pymatgen usage...
⚠️  Skipping dependent-repos/megnet/multifidelity/codes_for_plots/Figure4/plot_figure4_scatter_error.ipynb (AST parse error: invalid syntax (<unknown>, line 249))
🔍 Analyzing matgl for pymatgen usage...
⚠️  Skipping dependent-repos/matgl/examples/Training a QET Potential with PyTorch Lightning.ipynb (AST parse error: invalid syntax (<unknown>, line 12))
⚠️  Skipping dependent-repos/matgl/examples/Using LAMMPS with MatGL.ipynb (AST parse error: invalid syntax (<unknown>, line 58))


...

⚠️  Skipping dependent-repos/PaddleScience/jointContribution/IJCAI_2024/bju/download_dataset.ipynb (AST parse error: unexpected indent (<unknown>, line 12))
⚠️  Skipping dependent-repos/PaddleScience/jointContribution/IJCAI_2024/tenfeng/download_dataset.ipynb (AST parse error: unexpected indent (<unknown>, line 12))


<unknown>:316: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:430: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:361: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.


🔍 Analyzing PyXtal for pymatgen usage...
🔍 Analyzing pymatviz for pymatgen usage...
🔍 Analyzing all-atom-diffusion-transformer for pymatgen usage...
🔍 Analyzing quacc for pymatgen usage...
🔍 Analyzing doped for pymatgen usage...
⚠️  Skipping dependent-repos/doped/examples/plotting_customisation_tutorial.ipynb (AST parse error: invalid syntax (<unknown>, line 8))
⚠️  Skipping dependent-repos/doped/examples/chemical_potentials_tutorial.ipynb (AST parse error: invalid syntax (<unknown>, line 94))
⚠️  Skipping dependent-repos/doped/examples/GGA_workflow_tutorial.ipynb (AST parse error: invalid syntax (<unknown>, line 331))
⚠️  Skipping dependent-repos/doped/examples/generation_tutorial.ipynb (AST parse error: invalid syntax (<unknown>, line 8))
⚠️  Skipping dependent-repos/doped/examples/advanced_analysis_tutorial.ipynb (AST parse error: invalid syntax (<unknown>, line 135))
⚠️  Skipping dependent-repos/doped/examples/parsing_tutorial.ipynb (AST parse error: invalid syntax (<unknown>, line

<unknown>:329: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:329: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:346: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:346: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.


🔍 Analyzing matbench-discovery for pymatgen usage...
🔍 Analyzing CrystaLLM for pymatgen usage...
✅ Analysis finished.


In [ ]:
from collections import defaultdict

import plotly
import plotly.graph_objects as go

# Illegal/deprecated top-level PMG APIs
ILLEGAL_PMGPACKS: list[str] = [
    "Structure",
    "Molecule",
    "MPRester",
    "Composition",
    "Element",
]

# Manually filter some seldom used PMG subpackages
ILLEGAL_PMGPACKS += [
    "phonon",  # only used 1
    "transformations",  # only used 3
    "command_line",  # only used 2
]

# Subpackage color mapping
PMG_COLORS = {
    "core": "#3B9C9C",
    "analysis": "#6C5B7B",
    "io": "#355C7D",
    "entries": "#99B898",
    "symmetry": "#FFD3B6",
    "ext": "#E84A5F",
    "electronic_structure": "#F67280",
    "transformations": "#C06C84",
    "phonon": "#A8E6CF",
    "optimization": "#FFAAA5",
    "util": "#FF8B94",
    "command_line": "#0384fc",
    "alchemy": "#6203fc",
    "apps": "#343deb",
}


def hex_to_rgba(hex_color: str, alpha: float = 0.5) -> str:
    """Convert hex color to rgba string."""
    hex_color = hex_color.lstrip("#")
    red, green, blue = tuple(int(hex_color[idx : idx + 2], 16) for idx in (0, 2, 4))
    return f"rgba({red},{green},{blue},{alpha})"


# Aggregate usage data by subpackage
flows: list[tuple[str, str, int]] = []

for repo, result in results.items():
    usage = result["usage"]
    sub_counts = defaultdict(int)
    for api, count in usage.items():
        parts = api.split(".")
        if not parts or parts[0] != "pymatgen":
            continue
        if len(parts) >= 2 and parts[1] in ILLEGAL_PMGPACKS:
            continue

        subpkg = parts[1] if len(parts) >= 2 else None
        if subpkg:
            sub_counts[subpkg] += count
    for subpkg, total in sub_counts.items():
        flows.append((repo, subpkg, total))

# Build node labels: PMG packages on left, repos on right
subpackages: list[str] = list(PMG_COLORS.keys())
labels: list[str] = subpackages + repos
label_to_idx: dict[str, int] = {lbl: idx for idx, lbl in enumerate(labels)}

# Create flow from PMG packages (sources) to repos (targets)
sources: list[int] = [label_to_idx[sub] for _, sub, _ in flows]
targets: list[int] = [label_to_idx[repo] for repo, _, _ in flows]
values: list[float] = [val for _, _, val in flows]

# Assign colors
node_colors = [PMG_COLORS[lbl] if lbl in PMG_COLORS else "#F7B267" for lbl in labels]
link_colors = [hex_to_rgba(PMG_COLORS[sub], 0.5) for _, sub, _ in flows]

# Create Sankey diagram
assert len(labels) == len(subpackages) + len(repos)
assert len(labels) == len(node_colors)

assert all(index < len(subpackages) for index in sources)
assert all(index >= len(subpackages) for index in targets)
assert len(sources) == len(targets) == len(values) == len(link_colors)
fig = go.Figure(
    go.Sankey(
        arrangement="snap",
        node={
            "label": labels,
            "color": node_colors,
            "line": {"color": "rgba(0,0,0,0.1)", "width": 0.5},
        },
        link={
            "source": sources,
            "target": targets,
            "value": values,
            "color": link_colors,
        },
    )
)

fig.update_layout(
    font={"size": 20},
    height=800,
    width=600,
    margin={"l": 5, "r": 5, "t": 20, "b": 20},
)

fig.show()

# Fix ratio: https://github.com/plotly/Kaleido/issues/378
plotly.io.defaults.default_width = None
plotly.io.defaults.default_height = None

fig.write_image(
    Path.cwd().parents[1] / "paper" / "figs" / "dependent-usage-of-pmg.pdf",
    format="pdf",
)